In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from analyses.rust import contact
from pathlib import Path

In [ ]:
latpaths = {}
for replica in Path("../runs/cil_med-act/energy-10/").iterdir():
    rep = int(replica.name)
    latpaths[rep] = []
    for file in replica.joinpath("lattices").iterdir():
        latpaths[rep].append((
            file,
            str(file).replace("lattices", "act_lattices")
        ))

In [ ]:
def acts_to_df(acts):
    return pl.DataFrame({
        "label": ["c"] * len(acts.cell) + ["m"] * len(acts.medium) + ["s"] * len(acts.solid),
        "act": acts.cell + acts.medium + acts.solid,
    })

In [ ]:
klats = []
dfs = []
for r in range(0, 17):
    clat_path, alat_path = latpaths[0][100]
    clat, alat = pl.read_parquet(clat_path), pl.read_parquet(alat_path)
    res = contact.kernel_act(clat, alat, r, True)
    klat = res[1]
    klat = np.array(klat).reshape(500, 500)
    klats.append(klat)
    dfs.append(acts_to_df(res[0]).with_columns(radius=r))
klat_3d = np.array(klats)
rdf = pl.concat(dfs)

In [ ]:
px.imshow(klat_3d, animation_frame=0, width=800, height=800)

In [ ]:
px.violin(rdf, y="act", color="label", animation_frame="radius")

In [ ]:
cdf = rdf.group_by(["radius", "label"]).agg(
    mean=pl.col("act").mean()
).sort(
    "radius"
).pivot(
    on="label",
    index="radius", 
    values="mean"
).with_columns(
    diff=pl.col("m") - pl.col("c"),
    mdiff=pl.col("m") / pl.col("c")
).unpivot(
    index="radius"
)
px.bar(cdf, x="radius", y="value", color="variable", barmode="group")